In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

PROJ = Path(r"c:\Users\HP\Desktop\replication+contribution")
print("Project root:", PROJ)

Project root: c:\Users\HP\Desktop\replication+contribution


In [3]:
# ====================== 1. LOAD ORIGINAL .dta (ROBUST) ======================
print("\n" + "="*60)
print("LOADING ORIGINAL Saadaoui .dta")
print("="*60)

df_base = pd.read_stata(PROJ / "data" / "Saadaoui_2026_JCE.dta")

# ── Robust Period handling ─────────────────────────────────────────────
if pd.api.types.is_numeric_dtype(df_base['Period']):
    print("Period is numeric → converting from Stata %tm")
    base = pd.Timestamp('1960-01-01')
    df_base['date'] = df_base['Period'].apply(lambda m: base + pd.DateOffset(months=int(m)))
else:
    print("Period is already datetime-like")
    df_base['date'] = pd.to_datetime(df_base['Period'])

df_base = df_base.set_index('date').sort_index()
df_base.index = df_base.index.to_period('M').to_timestamp()

print(f"Base shape: {df_base.shape}")
print(f"Date range: {df_base.index[0]} to {df_base.index[-1]}")

# Compute differences (Stata style)
df_base['dllgop']  = df_base['llgop'].diff()
df_base['dl2lgop'] = df_base['l2lgop'].diff()


LOADING ORIGINAL Saadaoui .dta
Period is already datetime-like
Base shape: (386, 48)
Date range: 1990-01-01 00:00:00 to 2022-02-01 00:00:00


In [4]:
# ====================== 2. LOAD MACRO FEATURES ======================
print("\n" + "="*60)
print("LOADING MACRO FEATURE MATRIX (notebook 02)")
print("="*60)

df_macro = pd.read_csv(PROJ / "data" / "02_features" / "feature_matrix.csv", 
                       index_col=0, parse_dates=True)
df_macro.index = pd.to_datetime(df_macro.index).to_period('M').to_timestamp()
print(f"Macro shape: {df_macro.shape}")


LOADING MACRO FEATURE MATRIX (notebook 02)
Macro shape: (386, 68)


In [5]:
# ====================== 3. LOAD NLP FEATURES ======================
print("\n" + "="*60)
print("LOADING NLP FEATURE MATRIX A (notebook 03)")
print("="*60)

df_nlp = pd.read_csv(PROJ / "data" / "03_nlp" / "feature_matrix_nlp_A.csv", 
                     index_col=0, parse_dates=True)
df_nlp.index = pd.to_datetime(df_nlp.index).to_period('M').to_timestamp()
print(f"NLP shape: {df_nlp.shape}")


LOADING NLP FEATURE MATRIX A (notebook 03)
NLP shape: (386, 103)


In [6]:
# ====================== 4. MERGE ======================
print("\n" + "="*60)
print("MERGING CLEANLY")
print("="*60)

df_clean = df_base.copy()

# Macro merge (no lag)
df_clean = df_clean.join(df_macro, how='left', rsuffix='_macro')

# NLP merge with 1-month lag (important for no look-ahead)
df_nlp_lagged = df_nlp.shift(1)
df_clean = df_clean.join(df_nlp_lagged, how='left', rsuffix='_nlp')

print(f"Final clean matrix shape: {df_clean.shape}")


MERGING CLEANLY
Final clean matrix shape: (386, 221)


In [7]:
# ====================== 5. CLEANUP & SAVE ======================
forbidden = ['lbrent', 'cny_usd', 'em_fx_idx', 'reer', 'indpro', 'epu', 'bdi']
df_clean = df_clean.drop(columns=[c for c in forbidden if c in df_clean.columns], errors='ignore')

FINAL_PATH = PROJ / "data" / "final" / "feature_matrix_CLEAN.csv"
FINAL_PATH.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(FINAL_PATH)

print(f"\n✅ Saved: {FINAL_PATH}")


✅ Saved: c:\Users\HP\Desktop\replication+contribution\data\final\feature_matrix_CLEAN.csv


In [8]:
# ====================== 6. DIAGNOSTICS ======================
print("\n" + "="*60)
print("DIAGNOSTICS")
print("="*60)

print(f"d2pri-lpri correlation: {df_clean['d2pri'].corr(df_clean['lpri']):.4f}")

from statsmodels.api import OLS, add_constant
controls_base = ['llwip', 'dllgop', 'l2lwip', 'dl2lgop']
X = add_constant(df_clean[['d2pri'] + controls_base].dropna())
y = df_clean.loc[X.index, 'lpri']
model = OLS(y, X).fit()
fval = model.f_test("d2pri = 0").fvalue
print(f"Baseline first-stage F-stat: {fval:.2f}  (target ~236)")


DIAGNOSTICS
d2pri-lpri correlation: -0.0093
Baseline first-stage F-stat: 0.08  (target ~236)


In [9]:
# ====================== 4. MERGE WITH EXPLICIT ALIGNMENT CHECK ======================
print("\n" + "="*60)
print("MERGING WITH ALIGNMENT VERIFICATION")
print("="*60)

# Make sure all indices are identical before merging
print("Date alignment check:")
print("Base  dates match macro?", df_base.index.equals(df_macro.index))
print("Base  dates match NLP?  ", df_base.index.equals(df_nlp.index))

# Force reindex to be safe
common_dates = df_base.index.intersection(df_macro.index).intersection(df_nlp.index)

df_base_aligned  = df_base.loc[common_dates].copy()
df_macro_aligned = df_macro.loc[common_dates].copy()
df_nlp_aligned   = df_nlp.loc[common_dates].copy()

print(f"Common dates: {len(common_dates)} months")

df_clean = df_base_aligned.copy()

# Merge macro (no lag)
df_clean = df_clean.join(df_macro_aligned, how='left', rsuffix='_macro')

# Merge NLP **with 1-month lag**
df_nlp_lagged = df_nlp_aligned.shift(1)
df_clean = df_clean.join(df_nlp_lagged, how='left', rsuffix='_nlp')

print(f"Final clean matrix shape: {df_clean.shape}")

# Drop forbidden raw levels
forbidden = ['lbrent', 'cny_usd', 'em_fx_idx', 'reer', 'indpro', 'epu', 'bdi']
df_clean = df_clean.drop(columns=[c for c in forbidden if c in df_clean.columns], errors='ignore')

# ====================== DIAGNOSTICS ======================
print("\n" + "="*60)
print("CRITICAL DIAGNOSTICS")
print("="*60)

print(f"d2pri-lpri correlation: {df_clean['d2pri'].corr(df_clean['lpri']):.4f}")

from statsmodels.api import OLS, add_constant
base_controls = ['llwip', 'dllgop', 'l2lwip', 'dl2lgop']
X = add_constant(df_clean[['d2pri'] + base_controls].dropna())
y = df_clean.loc[X.index, 'lpri']
model = OLS(y, X).fit()
fval = model.f_test("d2pri = 0").fvalue
print(f"First-stage F-stat (baseline controls): {fval:.2f}")

# Compare first few rows of key variables
print("\nFirst 6 rows comparison (d2pri & lpri):")
print(df_clean[['d2pri', 'lpri']].head(6))


MERGING WITH ALIGNMENT VERIFICATION
Date alignment check:
Base  dates match macro? True
Base  dates match NLP?   True
Common dates: 386 months
Final clean matrix shape: (386, 221)

CRITICAL DIAGNOSTICS
d2pri-lpri correlation: -0.0093
First-stage F-stat (baseline controls): 0.08

First 6 rows comparison (d2pri & lpri):
            d2pri      lpri
date                       
1990-01-01   -0.1 -0.568825
1990-02-01   -0.2 -0.732668
1990-03-01    0.1 -0.808867
1990-04-01    0.1 -0.808867
1990-05-01    0.0 -0.808867
1990-06-01    0.0 -0.808867


In [11]:
import pandas as pd
from pathlib import Path

PROJ = Path(r"c:\Users\HP\Desktop\replication+contribution")

print("="*80)
print("SOURCE FILES INSPECTION")
print("="*80)

# Original .dta
df_orig = pd.read_stata(PROJ / "data" / "Saadaoui_2026_JCE.dta")

if pd.api.types.is_numeric_dtype(df_orig['Period']):
    base = pd.Timestamp('1960-01-01')
    df_orig['date'] = df_orig['Period'].apply(lambda m: base + pd.DateOffset(months=int(m)))
else:
    df_orig['date'] = pd.to_datetime(df_orig['Period'])

df_orig = df_orig.set_index('date').sort_index()
df_orig.index = df_orig.index.to_period('M').to_timestamp()

print(f"Original .dta : {df_orig.shape} | corr(d2pri,lpri) = {df_orig['d2pri'].corr(df_orig['lpri']):.4f}")

# Macro matrix
df_macro = pd.read_csv(PROJ / "data" / "02_features" / "feature_matrix.csv", 
                       index_col=0, parse_dates=True)
df_macro.index = pd.to_datetime(df_macro.index).to_period('M').to_timestamp()

has_d2pri = 'd2pri' in df_macro.columns
print(f"Macro matrix  : {df_macro.shape} | has d2pri/lpri = {has_d2pri}")
if has_d2pri:
    print(f"   corr in macro = {df_macro['d2pri'].corr(df_macro['lpri']):.4f}")

# NLP matrix
df_nlp = pd.read_csv(PROJ / "data" / "03_nlp" / "feature_matrix_nlp_A.csv", 
                     index_col=0, parse_dates=True)
df_nlp.index = pd.to_datetime(df_nlp.index).to_period('M').to_timestamp()

has_d2pri_nlp = 'd2pri' in df_nlp.columns
print(f"NLP matrix A  : {df_nlp.shape} | has d2pri/lpri = {has_d2pri_nlp}")
if has_d2pri_nlp:
    print(f"   corr in NLP   = {df_nlp['d2pri'].corr(df_nlp['lpri']):.4f}")

print("\nFirst 5 d2pri values:")
print("Orig  :", df_orig['d2pri'].head().tolist())
print("Macro :", df_macro['d2pri'].head().tolist() if has_d2pri else "N/A")
print("NLP   :", df_nlp['d2pri'].head().tolist() if has_d2pri_nlp else "N/A")

SOURCE FILES INSPECTION
Original .dta : (386, 48) | corr(d2pri,lpri) = -0.0093
Macro matrix  : (386, 68) | has d2pri/lpri = True
   corr in macro = -0.0093
NLP matrix A  : (386, 103) | has d2pri/lpri = True
   corr in NLP   = -0.0093

First 5 d2pri values:
Orig  : [-0.10000000149011612, -0.20000000298023224, 0.10000000149011612, 0.10000000149011612, 0.0]
Macro : [-0.1000000014901161, -0.2000000029802322, 0.1000000014901161, 0.1000000014901161, 0.0]
NLP   : [-0.1000000014901161, -0.2000000029802322, 0.1000000014901161, 0.1000000014901161, 0.0]


In [12]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJ = Path(r"c:\Users\HP\Desktop\replication+contribution")
print("Project root:", PROJ)

print("\n" + "="*80)
print("FULL DATA INSPECTION")
print("="*80)

# 1. Original .dta
print("\n1. ORIGINAL .dta")
df_orig = pd.read_stata(PROJ / "data" / "Saadaoui_2026_JCE.dta")

if pd.api.types.is_numeric_dtype(df_orig['Period']):
    base = pd.Timestamp('1960-01-01')
    df_orig['date'] = df_orig['Period'].apply(lambda m: base + pd.DateOffset(months=int(m)))
else:
    df_orig['date'] = pd.to_datetime(df_orig['Period'])

df_orig = df_orig.set_index('date').sort_index()
df_orig.index = df_orig.index.to_period('M').to_timestamp()

print(f"Shape: {df_orig.shape}")
print(f"corr(d2pri, lpri) = {df_orig['d2pri'].corr(df_orig['lpri']):.4f}")
print("First 6 d2pri:", df_orig['d2pri'].head(6).tolist())
print("Any duplicate dates?", df_orig.index.duplicated().any())

# 2. Macro feature matrix
print("\n2. MACRO FEATURE MATRIX (02_features)")
df_macro = pd.read_csv(PROJ / "data" / "02_features" / "feature_matrix.csv", 
                       index_col=0, parse_dates=True)
df_macro.index = pd.to_datetime(df_macro.index).to_period('M').to_timestamp()

print(f"Shape: {df_macro.shape}")
print(f"corr(d2pri, lpri) = {df_macro.get('d2pri', pd.Series()).corr(df_macro.get('lpri')):.4f}")
print("d2pri in columns?", 'd2pri' in df_macro.columns)
print("Duplicate dates?", df_macro.index.duplicated().any())

# 3. NLP matrix A
print("\n3. NLP MATRIX A (03_nlp)")
df_nlp = pd.read_csv(PROJ / "data" / "03_nlp" / "feature_matrix_nlp_A.csv", 
                     index_col=0, parse_dates=True)
df_nlp.index = pd.to_datetime(df_nlp.index).to_period('M').to_timestamp()

print(f"Shape: {df_nlp.shape}")
print(f"corr(d2pri, lpri) = {df_nlp.get('d2pri', pd.Series()).corr(df_nlp.get('lpri')):.4f}")
print("Duplicate dates?", df_nlp.index.duplicated().any())

# 4. Raw files quick scan (optional but useful)
print("\n4. RAW FILES IN 02_features/raw/")
raw_dir = PROJ / "data" / "02_features" / "raw"
for f in sorted(raw_dir.glob("*.*")):
    try:
        if f.suffix == '.csv':
            tmp = pd.read_csv(f, nrows=5)
            print(f"  {f.name:25s} → {tmp.shape[1]} cols, first date col: {tmp.columns[0] if len(tmp.columns)>0 else 'N/A'}")
    except:
        print(f"  {f.name:25s} → read error")

Project root: c:\Users\HP\Desktop\replication+contribution

FULL DATA INSPECTION

1. ORIGINAL .dta
Shape: (386, 48)
corr(d2pri, lpri) = -0.0093
First 6 d2pri: [-0.10000000149011612, -0.20000000298023224, 0.10000000149011612, 0.10000000149011612, 0.0, 0.0]
Any duplicate dates? False

2. MACRO FEATURE MATRIX (02_features)
Shape: (386, 68)
corr(d2pri, lpri) = -0.0093
d2pri in columns? True
Duplicate dates? False

3. NLP MATRIX A (03_nlp)
Shape: (386, 103)
corr(d2pri, lpri) = -0.0093
Duplicate dates? False

4. RAW FILES IN 02_features/raw/
  baa10y.csv                → 2 cols, first date col: observation_date
  bdi_clean.csv             → 2 cols, first date col: date
  brent_fred.csv            → 2 cols, first date col: observation_date
  cny_usd.csv               → 2 cols, first date col: observation_date
  dxy.csv                   → 2 cols, first date col: observation_date
  em_fx_idx.csv             → 2 cols, first date col: observation_date
  em_oas.csv                → 2 cols, first 

In [13]:
import pandas as pd
from pathlib import Path

PROJ = Path(r"c:\Users\HP\Desktop\replication+contribution")

df = pd.read_stata(PROJ / "data" / "Saadaoui_2026_JCE.dta")

# Robust date conversion
if pd.api.types.is_numeric_dtype(df['Period']):
    base = pd.Timestamp('1960-01-01')
    df['date'] = df['Period'].apply(lambda m: base + pd.DateOffset(months=int(m)))
else:
    df['date'] = pd.to_datetime(df['Period'])

df = df.set_index('date').sort_index()
df.index = df.index.to_period('M').to_timestamp()

print("=== ORIGINAL .dta DEEP STATS ===")
print(f"Shape: {df.shape}")
print(f"Date range: {df.index[0]} → {df.index[-1]}")
print(f"Duplicate dates: {df.index.duplicated().sum()}")

print("\nKey variables summary:")
print(df[['lpri', 'd2pri', 'lwti']].describe().round(4))

print("\nCorrelation matrix:")
print(df[['lpri', 'd2pri', 'lwti']].corr().round(4))

print("\nFirst 10 d2pri vs lpri:")
print(df[['d2pri', 'lpri']].head(10))

print("\nLast 10 d2pri vs lpri:")
print(df[['d2pri', 'lpri']].tail(10))

# Check if d2pri looks like a proper second difference
print("\nd2pri basic stats:")
print(f"Mean: {df['d2pri'].mean():.6f}")
print(f"Std : {df['d2pri'].std():.6f}")
print(f"Min : {df['d2pri'].min():.4f} | Max: {df['d2pri'].max():.4f}")

=== ORIGINAL .dta DEEP STATS ===
Shape: (386, 48)
Date range: 1990-01-01 00:00:00 → 2022-02-01 00:00:00
Duplicate dates: 0

Key variables summary:
           lpri     d2pri      lwti
count  386.0000  386.0000  386.0000
mean     0.0217   -0.0003    3.0278
std      1.2678    0.3379    0.4747
min     -2.8130   -2.2000    1.8660
25%     -0.7327   -0.1000    2.6631
50%      0.1987    0.0000    2.9734
75%      1.0160    0.1000    3.4228
max      1.9093    2.2000    4.1205

Correlation matrix:
         lpri   d2pri    lwti
lpri   1.0000 -0.0093  0.2586
d2pri -0.0093  1.0000  0.0147
lwti   0.2586  0.0147  1.0000

First 10 d2pri vs lpri:
            d2pri      lpri
date                       
1990-01-01   -0.1 -0.568825
1990-02-01   -0.2 -0.732668
1990-03-01    0.1 -0.808867
1990-04-01    0.1 -0.808867
1990-05-01    0.0 -0.808867
1990-06-01    0.0 -0.808867
1990-07-01    0.0 -0.808867
1990-08-01    0.0 -0.808867
1990-09-01    0.0 -0.808867
1990-10-01    0.0 -0.808867

Last 10 d2pri vs lpri:
   

In [14]:
import pandas as pd
from pathlib import Path
import pyreadstat

PROJ = Path(r"C:\Users\HP\Desktop\replication+contribution")

# Load original .dta properly
df, meta = pyreadstat.read_dta(PROJ / "data" / "Saadaoui_2026_JCE.dta")
print("Raw .dta columns:", df.columns.tolist())
print("\nFirst 5 rows of Period, pri, lpri, d2pri, lwti:")
print(df[['Period', 'pri', 'lpri', 'd2pri', 'lwti']].head())

# Check if d2pri is actually second difference of pri
df['pri_diff1'] = df['pri'].diff()
df['pri_diff2'] = df['pri_diff1'].diff()
print("\nDoes d2pri == second diff of pri?")
print((df['d2pri'].dropna() == df['pri_diff2'].dropna()).all())

Raw .dta columns: ['lwip', 'lgop', 'lwti', 'lpri', 'pri', 'date', 'u_wip', 'u_gop', 'pri_jp', 'pri_aus', 'pri_cds', 'pri_fra', 'pri_ger', 'pri_india', 'pri_indo', 'pri_pak', 'pri_rus', 'pri_vn', 'pri_uk', 'Period', 'lpri_jp', 'dlpri_jp', 'lpri_aus', 'dlpri_aus', 'lpri_cds', 'dlpri_cds', 'lpri_fra', 'dlpri_fra', 'lpri_ger', 'dlpri_ger', 'lpri_india', 'dlpri_india', 'lpri_indo', 'dlpri_indo', 'lpri_pak', 'dlpri_pak', 'lpri_rus', 'dlpri_rus', 'lpri_vn', 'dlpri_vn', 'lpri_uk', 'dlpri_uk', 'dlpri', 'llwip', 'llgop', 'l2lwip', 'l2lgop', 'd2pri', 'd2pri_jp']

First 5 rows of Period, pri, lpri, d2pri, lwti:
   Period  pri      lpri  d2pri      lwti
0   360.0 -0.6 -0.568825   -0.1  2.876816
1   361.0 -0.8 -0.732668   -0.2  2.849079
2   362.0 -0.9 -0.808867    0.1  2.764880
3   363.0 -0.9 -0.808867    0.1  2.668327
4   364.0 -0.9 -0.808867    0.0  2.648035

Does d2pri == second diff of pri?


ValueError: Can only compare identically-labeled Series objects

In [15]:
import pandas as pd
import numpy as np
from pathlib import Path
import pyreadstat

PROJ = Path(r"C:\Users\HP\Desktop\replication+contribution")
df, meta = pyreadstat.read_dta(PROJ / "data" / "Saadaoui_2026_JCE.dta")

# Reconstruct d2pri from pri to verify
df['pri_diff1'] = df['pri'].diff()
df['pri_diff2'] = df['pri_diff1'].diff()

# Proper comparison (align indices first)
aligned = pd.concat([df['d2pri'], df['pri_diff2']], axis=1).dropna()
print(f"\nDoes d2pri == second diff of pri? {(aligned['d2pri'] == aligned['pri_diff2']).all()}")
print(f"Max absolute difference: {(aligned['d2pri'] - aligned['pri_diff2']).abs().max():.10f}")

# Check correlation in RAW .dta (before any date conversion)
print(f"\nRAW .dta correlation (pri, d2pri): {df['pri'].corr(df['d2pri']):.4f}")
print(f"RAW .dta correlation (lpri, d2pri): {df['lpri'].corr(df['d2pri']):.4f}")

# Now convert date and check if conversion breaks it
base = pd.Timestamp('1960-01-01')
df['date'] = df['Period'].apply(lambda m: base + pd.DateOffset(months=int(m)))
df = df.set_index('date').sort_index()

print(f"\nAfter date conversion — correlation (lpri, d2pri): {df['lpri'].corr(df['d2pri']):.4f}")
print(f"Index is sorted? {df.index.is_monotonic_increasing}")
print(f"Any duplicate dates? {df.index.duplicated().any()}")


Does d2pri == second diff of pri? False
Max absolute difference: 0.0000000477

RAW .dta correlation (pri, d2pri): -0.0097
RAW .dta correlation (lpri, d2pri): -0.0093

After date conversion — correlation (lpri, d2pri): -0.0093
Index is sorted? True
Any duplicate dates? False


In [16]:
import pandas as pd
import numpy as np
from pathlib import Path
from statsmodels.api import OLS, add_constant
from sklearn.ensemble import RandomForestRegressor
import warnings
warnings.filterwarnings('ignore')

PROJ = Path(r"C:\Users\HP\Desktop\replication+contribution")
df = pd.read_csv(PROJ / "data" / "final" / "feature_matrix_nlp_FINAL.csv", 
                 index_col=0, parse_dates=True)
df.index = pd.to_datetime(df.index).to_period('M').to_timestamp()

Y, T, Z = "lwti", "lpri", "d2pri"

# ==================== STEP 1: FIRST STAGE (with lags of lpri) ====================
# These are PREDICTORS, not confounders
first_stage_controls = ['llwip', 'dllgop', 'l2lwip', 'dl2lgop', 'L1_lpri', 'L2_lpri']

# Create lags if missing
for lag in [1, 2]:
    col = f"L{lag}_lpri"
    if col not in df.columns:
        df[col] = df['lpri'].shift(lag)

df_fs = df[['lpri', 'd2pri'] + first_stage_controls].dropna()

X_fs = add_constant(df_fs[['d2pri'] + first_stage_controls])
y_fs = df_fs['lpri']

fs_model = OLS(y_fs, X_fs).fit()
print(f"First-stage F-stat on d2pri: {fs_model.f_test('d2pri = 0').fvalue:.2f}")

# Save residuals (control function)
df_fs['lpri_hat'] = fs_model.fittedvalues
df_fs['v_hat'] = fs_model.resid  # this is the "cleaned" treatment

# ==================== STEP 2: SECOND STAGE (causal) ====================
# Controls for outcome: baseline + lags of lwti + other features (NO lags of lpri)
second_stage_controls = ['llwip', 'dllgop', 'l2lwip', 'dl2lgop', 'L1_lwti', 'L2_lwti']

# Add lwti lags if missing
for lag in [1, 2]:
    col = f"L{lag}_lwti"
    if col not in df.columns:
        df[col] = df['lwti'].shift(lag)

# Merge the control function back
df['v_hat'] = df_fs['v_hat']

def run_cf_horizon(h):
    lead_col = f"lwti_h{h}"
    df_temp = df.copy()
    df_temp[lead_col] = df_temp['lwti'].shift(-h)
    
    # Second stage: Y_h ~ lpri + v_hat + controls
    # v_hat is the control function (first-stage residual)
    # Including v_hat removes endogeneity bias
    ss_vars = [lead_col, 'lpri', 'v_hat'] + second_stage_controls
    data = df_temp[ss_vars].dropna()
    if len(data) < 100:
        return None
    
    X = add_constant(data[['lpri', 'v_hat'] + second_stage_controls])
    y = data[lead_col]
    
    model = OLS(y, X).fit()
    coef = model.params['lpri']
    se = model.bse['lpri']
    
    print(f"h={h:2d} | coef={coef:.4f} | se={se:.4f} | N={len(data)}")
    return {"h": h, "coef": coef, "se": se, "n": len(data)}

# Run
results = []
for h in [0, 6, 12, 24, 36, 48]:
    r = run_cf_horizon(h)
    if r:
        results.append(r)

df_irf = pd.DataFrame(results)
df_irf.to_csv(PROJ / "results" / "cf_irf_correct.csv", index=False)
print("\nSaved cf_irf_correct.csv")
print(df_irf.round(4))

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\HP\\Desktop\\replication+contribution\\data\\final\\feature_matrix_nlp_FINAL.csv'